# 2. Recolección de Datos - Web Scraping (Wikipedia)

**Objetivo:** completar los datos de la API con información histórica de la tabla de Wikipedia
*"List of Falcon 9 and Falcon Heavy launches"*, que tiene el detalle de cada lanzamiento
(fecha, versión del booster, sitio, carga útil, órbita, cliente, resultado y aterrizaje).

**Flujo de trabajo:**

1. Pedir el HTML de una **versión fija** (snapshot) de la página de Wikipedia con `requests`,
   para que el dataset no cambie si alguien edita el artículo después.
2. Parsear el HTML con `BeautifulSoup` y ubicar la tabla `wikitable plainrowheaders collapsible`
   con el historial de lanzamientos.
3. Extraer los nombres de columna a partir de los `<th>` del encabezado (limpiando `<br>`, links y
   referencias `[n]`).
4. Recorrer cada fila (`<tr>`) de la tabla: si la primera celda (`<th>`) es un número de vuelo,
   es una fila de datos válida.
5. Por cada fila, extraer: fecha, hora, versión del booster, sitio de lanzamiento, payload,
   masa del payload, órbita, cliente, resultado del lanzamiento y resultado del aterrizaje.
6. Armar un DataFrame y exportarlo a `data/raw/spacex_web_scraped.csv`.

> **Nota:** este notebook necesita conexión a internet para descargar la página de Wikipedia.

In [ ]:
import re
import unicodedata
import requests
import pandas as pd
from bs4 import BeautifulSoup

pd.set_option('display.max_columns', None)

## Paso 1: descargar el HTML (snapshot fijo del artículo)

Usamos una revisión específica (`oldid`) del artículo de Wikipedia para que el contenido no
cambie con el tiempo y el análisis sea reproducible.

In [ ]:
static_url = ("https://en.wikipedia.org/w/index.php?title="
              "List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922")

response = requests.get(static_url)
print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, 'html.parser')
soup.title

## Paso 2: funciones auxiliares de parseo

Cada función limpia el contenido de una celda de la tabla para quedarnos con el dato limpio.

In [ ]:
def extract_column_from_header(row):
    """Limpia un <th> de encabezado: saca <br>, links y notas al pie [n]."""
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not column_name.strip().isdigit():
        return column_name.strip()


def date_time(table_cells):
    """Devuelve [fecha, hora] a partir de la celda de fecha/hora."""
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]


def booster_version(table_cells):
    """Texto de version+serie del booster, sin las referencias [n] entre corchetes."""
    text = table_cells.get_text(separator='', strip=True)
    text = re.sub(r'\[[^\]]*\]', '', text)
    return text.strip()


def landing_status(table_cells):
    """Primer string de la celda de aterrizaje (ej. 'Success', 'Failure', 'No attempt')."""
    strings = list(table_cells.strings)
    return strings[0] if strings else None


def get_mass(table_cells):
    """Masa del payload en texto (ej. '5000 kg'), o 0 si la celda está vacía."""
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        return mass[0:mass.find("kg") + 2]
    return 0


def first_link_text(cell):
    """Texto del primer link de la celda si existe; si no, el texto plano de la celda."""
    if cell.a and cell.a.string:
        return cell.a.string
    text = cell.get_text(strip=True)
    return text if text else None

## Paso 3: nombres de columna a partir de los `<th>` del encabezado

In [ ]:
html_tables = soup.find_all('table', "wikitable plainrowheaders collapsible")
first_launch_table = html_tables[0]

column_names = []
for row in first_launch_table.find_all('th'):
    name = extract_column_from_header(row)
    if name is not None and len(name) > 0:
        column_names.append(name)

column_names

## Paso 4: recorrer las filas y extraer los datos de cada lanzamiento

In [ ]:
launch_dict = {
    'Flight No.': [],
    'Launch site': [],
    'Payload': [],
    'Payload mass': [],
    'Orbit': [],
    'Customer': [],
    'Launch outcome': [],
    'Version Booster': [],
    'Booster landing': [],
    'Date': [],
    'Time': [],
}

extracted_row = 0
for table in html_tables:
    for rows in table.find_all("tr"):
        flag = False
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        row = rows.find_all('td')
        if not flag:
            continue

        extracted_row += 1
        launch_dict['Flight No.'].append(flight_number)

        datatimelist = date_time(row[0])
        launch_dict['Date'].append(datatimelist[0].strip(','))
        launch_dict['Time'].append(datatimelist[1] if len(datatimelist) > 1 else None)

        bv = booster_version(row[1])
        if not bv:
            bv = first_link_text(row[1])
        launch_dict['Version Booster'].append(bv)

        launch_dict['Launch site'].append(first_link_text(row[2]))
        launch_dict['Payload'].append(first_link_text(row[3]))
        launch_dict['Payload mass'].append(get_mass(row[4]))
        launch_dict['Orbit'].append(first_link_text(row[5]))
        launch_dict['Customer'].append(first_link_text(row[6]))

        launch_outcome = list(row[7].strings)[0].strip()
        launch_dict['Launch outcome'].append(launch_outcome)

        launch_dict['Booster landing'].append(landing_status(row[8]))

print(f"Filas extraidas: {extracted_row}")

## Paso 5: armar el DataFrame y revisarlo

In [ ]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
df.head()

In [ ]:
df.shape

## Paso 6: exportar el dataset

In [ ]:
df.to_csv('../data/raw/spacex_web_scraped.csv', index=False)

## Resumen

Este dataset scrapeado de Wikipedia complementa al de la API con datos históricos más detallados
por lanzamiento (incluye lanzamientos del Falcon 1). En el siguiente notebook (**03 - Data
Wrangling**) vamos a limpiar, combinar y etiquetar los datos para dejarlos listos para el
análisis y el modelo de Machine Learning.